### Introduction
The way to run this is to use the enviroment.yml to create a conda environment.
You can either run main or all the numbered cells individually its the same code with minor changes to account for using multiple notebooks.
The structure is as follows:
```
├── README.md
├── Environment.yml           # Dependencies for Conda
├── main.ipynb
├── 1_Data_Preparation.ipynb
├── 2_KPIs.ipynb
├── 3_Site_Classification.ipynb
├── 4_Clustering.ipynb
├── 5_Regression.ipynb
├── intermediate_data         # to make multiple notebooks work
└── Data_Share/               # Standardized folder name (no spaces)
    ├── Charging_Sessions.csv
    └── Weather_Burbank_Airport.csv
```

The main file are structured like this the single files are analogous
1. Data Setup and Cleaning: Initializing the environment and preprocessing the raw CSV files.
2. KPIs: Calculation and visualization of the three primary Key Performance Indicators.
3. Site Classification: Determining and distinguishing between Private and Public sites.
4. Clustering: Adding further parameters needed for clustering and determining a good number of clusters.
5. Regression: Training the regression model and adding further parameters to aid with the training


### 1. Data Setup and Cleaning
In this first part we do the following:
* Load CSVs into DataFrames 
* expand user input fields 
 * standardize timezones
* Remove redundant columns and duplicate records
* Interpolate missing values and sync weather data with charging sessions based on overlapping timeframes
* Note:  Anything relating to the weather data will not be used why will be explained in the regression part

In [1]:
import os
# This is to avoid warnings on windows when running k-means clustering
os.environ['OMP_NUM_THREADS'] = '4'
import pandas as pd
import ast


In [2]:
def csv_to_pandas(filepath):
    df = pd.read_csv(filepath)
    return df


In [3]:
# load and convert to pandas
charging_sessions = csv_to_pandas("./Data Share/charging_sessions.csv")
weather_burbank_airport = csv_to_pandas("./Data Share/weather_burbank_airport.csv")

In [4]:
# expand userinput data
def parse_list(cell):
    if pd.isna(cell):
        return [] 
    return ast.literal_eval(cell)

charging_sessions["parsed"] = charging_sessions["userInputs"].apply(parse_list)

charging_sessions["last"] = charging_sessions["parsed"].apply(
    lambda lst: lst[-1] if len(lst) > 0 else None
)

expanded_df = pd.json_normalize(charging_sessions["last"])
expanded_df = expanded_df.drop(columns= "userID")
charging_sessions = pd.concat([charging_sessions, expanded_df], axis=1)



In [5]:
# convert to local time, the timezone for all the values is America/Los_Angeles

time_cols = [
    'connectionTime', 
    'disconnectTime', 
    'doneChargingTime', 
    'modifiedAt', 
    'requestedDeparture'
]

for col in time_cols:
    charging_sessions[col] = (
        pd.to_datetime(charging_sessions[col], utc=True, errors='coerce') 
        .dt.tz_convert('America/Los_Angeles')                              
        .dt.tz_localize(None)                                              
    )

In [6]:
# remove duplicate rows and useless columns
charging_sessions = charging_sessions[[
        "id",
        "connectionTime",
        "disconnectTime",
        "doneChargingTime",
        "kWhDelivered",
        "sessionID",
        "siteID",
        "stationID",
        "userID",
        "WhPerMile",
        "kWhRequested",
        "milesRequested",
        "minutesAvailable",
        "modifiedAt",
        "requestedDeparture"
    ]]
charging_sessions= charging_sessions.drop_duplicates(keep='first')

# rename columns to snake_case
charging_sessions = charging_sessions.rename(columns={
    'connectionTime': 'connection_time',
    'disconnectTime': 'disconnect_time',
    'doneChargingTime': 'done_charging_time',
    'kWhDelivered': 'kwh_delivered',
    'sessionID': 'session_id',
    'siteID': 'site_id',
    'stationID': 'station_id',
    'userID': 'user_id',
    'WhPerMile': 'wh_per_mile',
    'kWhRequested': 'kwh_requested',
    'milesRequested': 'miles_requested',
    'minutesAvailable': 'minutes_available',
    'modifiedAt': 'modified_at',
    'requestedDeparture': 'requested_departure',
})

In [7]:
# add missing values in temperature data by interpolation
numeric_cols = ['temperature', 'felt_temperature', 'windspeed', 'pressure','cloud_cover']
weather_burbank_airport[numeric_cols] = weather_burbank_airport[numeric_cols].interpolate(method='linear')

In [8]:
# only get the data which overlaps for weather and charging this isn't used
weather_burbank_airport['timestamp'] = pd.to_datetime(weather_burbank_airport['timestamp'])

cs_min = charging_sessions['connection_time'].min()
cs_max = charging_sessions['connection_time'].max()

weather_min = weather_burbank_airport['timestamp'].min()
weather_max = weather_burbank_airport['timestamp'].max()

overlap_start = max(cs_min, weather_min)
overlap_end = min(cs_max, weather_max)

charging_sessions_overlap = charging_sessions[
    (charging_sessions['connection_time'] >= overlap_start) &
    (charging_sessions['connection_time'] <= overlap_end)
]

weather_burbank_airport_overlap = weather_burbank_airport[
    (weather_burbank_airport['timestamp'] >= overlap_start) &
    (weather_burbank_airport['timestamp'] <= overlap_end)
]
weather_burbank_airport['timestamp'].max()

Timestamp('2021-01-01 07:53:00')

In [9]:
# Save intermediate data for subsequent notebooks
import os
os.makedirs('intermediate_data', exist_ok=True)
charging_sessions.to_parquet('intermediate_data/charging_sessions.parquet')
weather_burbank_airport.to_parquet('intermediate_data/weather_burbank_airport.parquet')